# Geological Surface Accuracy





# 0.1 Carica il workspace



eseguire (play) la cella "CARICA SPAZIO DI LAVORO" solo all'apertura del notebook (altrimenti si ricarica lo spazio di lavoro di default cancellando le modifiche):



- verr� caricato l'ambiente di lavoro e la cartella "cartella_files" dove andranno messi i file GOCAD delle superfici che costituiscono il modello 3D (.ts) e gli shape file che contengono le tracce delle sezioni sismiche



  e e i punti relativi ai pozzi 





In [ ]:
# ### CARICA SPAZIO DI LAVORO



import os

import shutil



# Percorso base

base_path = '/content'

repo_path = os.path.join(base_path, 'GeoSurface_Accuracy')



# Funzione per pulire completamente la directory

def clean_repo_directory():

    try:

        # Rimuovi la directory se esiste

        if os.path.exists(repo_path):

            shutil.rmtree(repo_path)

            print(f"Directory {repo_path} rimossa")

    except Exception as e:

        print(f"Errore nella rimozione della directory: {e}")



# Pulisci la directory

clean_repo_directory()



# Cambia nella directory base

os.chdir(base_path)



# Clona il repository

!git clone https://github.com/BaterHub/GeoSurface_Accuracy.git



# Cambia nella directory del repository

%cd GeoSurface_Accuracy





# 0.2 Carica i file nella cartella "cartella_files"



Trascinare i file* del pacchetto costituente il modello 3D nella cartella "working_files_folder"



*NB andranno caricati i seguenti file:



- horizons.ts (deve contenere tutte le geometrie delle superfici)



- shapefile delle tracce di sezioni geologiche e linee sismiche utilizzate per la costruzione della superficie





# 0.3 Eseguire lo script



- Posizionarsi nella cella "LANCIA LO SCRIPT" e eseguire il RUN con "ctrl + F10" oppure dal men� "Runtime > Run cell and below/Esegui questa cella e quelle sottostanti"

- Al termine del RUN verranno generati gli output e un log_file all'interno della working_files_folder.





# 1. Importa librerie e funzioni





In [ ]:
# ### LANCIA LO SCRIPT



## Importa librerie necessarie

import pandas as pd

import geopandas as gpd

import numpy as np

import matplotlib.pyplot as plt

from matplotlib.colors import LinearSegmentedColormap

from pyproj import Proj, transform

from scipy.spatial import cKDTree

from scipy.interpolate import griddata

from sklearn.preprocessing import MinMaxScaler

import os

import re

from pathlib import Path



#############################################################################################

## Importa funzioni

import importlib # modulo per il reload delle funzioni



# Reimporta i moduli originali

import files_utils



# Ricarica forzata di ciascun modulo

importlib.reload(files_utils)



# Reimporta le funzioni dai moduli ricaricati

from files_utils import *

#############################################################################################



# Percorso cartella

folder_name = "working_files_folder"

input_dir = os.path.abspath(folder_name)

output_dir = "output_results"





In [ ]:
# ### Impostazione delle cartella di lavoro

working_dir = "working_files_folder"

output_dir = "output_results"

crs = 'EPSG:6708'





In [ ]:
# Main Function ed esecuzione procedura
def main(working_dir=working_dir):
    print("Avvio dell'analisi dei dati geologici...")

    if not os.path.exists(working_dir):
        print(f"La cartella {working_dir} non esiste. Creazione in corso...")
        os.makedirs(working_dir)
        print(f"Cartella {working_dir} creata. Inserisci i file GOCAD .ts e gli shapefile nella cartella.")
        return None

    print(f"File presenti nella cartella {working_dir}:")
    for file in os.listdir(working_dir):
        print(f"  - {file}")

    surface_name = get_surface_name(working_dir)
    mapping = ensure_mapping_file(working_dir, surface_name)

    vertices, triangles = process_gocad_file(working_dir)
    wells_shp = read_wells_shapefile(working_dir) if mapping.get('use_wells', True) else None
    sections_shp = read_sections_shapefile(working_dir) if mapping.get('use_sections', True) else None

    has_vertices = vertices is not None and len(vertices) > 0
    has_triangles = triangles is not None and len(triangles) > 0
    has_wells = wells_shp is not None and not wells_shp.empty if wells_shp is not None else False
    has_sections = sections_shp is not None and not sections_shp.empty if sections_shp is not None else False

    print(f"Stato dei dati (superficie: {surface_name}):")
    print(f"  - Vertices: {'Presenti' if has_vertices else 'Assenti'} - {len(vertices) if has_vertices else 0} vertici")
    print(f"  - Triangoli: {'Presenti' if has_triangles else 'Assenti'} - {len(triangles) if has_triangles else 0} triangoli")
    print(f"  - Pozzi: {'Presenti' if has_wells else 'Assenti'}")
    print(f"  - Sezioni: {'Presenti' if has_sections else 'Assenti'}")

    grid_points = None
    grid_weights = None
    if has_vertices:
        acc_outputs = generate_accuracy_outputs(vertices, wells_shp, sections_shp, output_dir,
                                                use_wells=has_wells, use_sections=has_sections,
                                                grid_spacing=5000, line_step=2000)
        grid_points = acc_outputs.get('grid_points')
        grid_weights = acc_outputs.get('weights')
        print("Calcolo accuratezza orizzontale completato (CSV/PNG in output_results).")
    else:
        print("Nessun vertice disponibile per calcolare la griglia di valutazione.")

    if has_vertices or has_wells or has_sections:
        try:
            fig = visualize_data(vertices, triangles, wells_shp, sections_shp, apply_smoothing=False,
                                 smoothing_iterations=3, smoothing_factor=0.2, crs='EPSG:6708',
                                 output_filename='model_dataset.png')
            print("Visualizzazione completata con successo.")
        except Exception as e:
            print(f"Errore durante la visualizzazione: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("Non ci sono dati da visualizzare. Verifica che i file siano presenti nella cartella di lavoro.")

    print("Analisi completata.")

    return {
        'surface_name': surface_name,
        'vertices': vertices,
        'triangles': triangles,
        'wells': wells_shp,
        'sections': sections_shp,
        'grid_points': grid_points,
        'horizontal_weights': grid_weights
    }

if __name__ == "__main__":
    data = main()

